# Emotion Classification using DistilBERT
Complete fine-tuning notebook.

In [1]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [2]:
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer,AutoModelForSequenceClassification,TrainingArguments,Trainer,pipeline


In [5]:
dataset=load_dataset("dair-ai/emotion")
print(dataset)
print(dataset["train"][0])
print(dataset["train"].features["label"])

README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

split/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.03MB            

split/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

split/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  127kB            

split/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

split/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})
{'text': 'i didnt feel humiliated', 'label': 0}
ClassLabel(names=['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'])


In [6]:
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(example["text"],truncation=True,padding="max_length",max_length=128)

tokenized_dataset=dataset.map(tokenize,batched=True)
tokenized_dataset.set_format(type="torch",columns=["input_ids","attention_mask","label"])

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [7]:
model=AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased",num_labels=6)

accuracy=evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits,labels=eval_pred
    predictions=np.argmax(logits,axis=1)
    return accuracy.compute(predictions=predictions,references=labels)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
training_args=TrainingArguments(
output_dir="./emotion_model",
learning_rate=2e-5,
num_train_epochs=3,
per_device_train_batch_size=16,
per_device_eval_batch_size=16,
eval_strategy="epoch",
save_strategy="epoch",
logging_steps=50,
report_to="none",
optim="adamw_torch_xla"
)

trainer=Trainer(
model=model,
args=training_args,
train_dataset=tokenized_dataset["train"],
eval_dataset=tokenized_dataset["validation"],
compute_metrics=compute_metrics
)

In [12]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.241591,0.193034,0.926500
2,0.130496,0.166971,0.933500
3,0.087547,0.153117,0.939000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.24724539001782736, metrics={'train_runtime': 357.0203, 'train_samples_per_second': 134.446, 'train_steps_per_second': 8.403, 'total_flos': 1589722177536000.0, 'train_loss': 0.24724539001782736, 'epoch': 3.0})

In [13]:
print(trainer.evaluate())

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Training Loss,Validation Loss,Epoch,Accuracy
0.087547,0.153117,3,0.939000


{'eval_loss': 0.1531165987253189, 'eval_accuracy': 0.939}


In [14]:
trainer.save_model("emotion_classifier")
tokenizer.save_pretrained("emotion_classifier")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('emotion_classifier/tokenizer_config.json',
 'emotion_classifier/tokenizer.json')

In [15]:
classifier=pipeline("text-classification",model="emotion_classifier",tokenizer="emotion_classifier")
label_map={0:"Sadness",1:"Joy",2:"Love",3:"Anger",4:"Fear",5:"Surprise"}
text="I finally got my dream job!"
result=classifier(text)[0]
label_id=int(result["label"].split("_")[-1])
print(result)
print(label_map[label_id])

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'label': 'LABEL_1', 'score': 0.9951580166816711}
Joy
